# プロジェクト概要
- NBAデータの決定木分類を行いました
- NBAデータのランダムフォレスト分類を行いました
- pandas、matplotlib、numpy、sklearnライブラリを使用しました

In [19]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# 1. 決定木

### データの読み込みと欠損値の確認など

In [3]:
nba_22_23 = pd.read_csv('nba_22_23.csv')
nba_22_23.head()

,Rk,Player,Pos,Age,Tm,G,GS,MP,FG,FGA,...,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS
0,1,Precious Achiuwa,C,23,TOR,49,11,21.1,3.4,7.2,...,0.686,1.8,4.1,5.9,1.0,0.6,0.6,1.1,1.9,8.9
1,2,Steven Adams,C,29,MEM,42,42,27.0,3.7,6.3,...,0.364,5.1,6.5,11.5,2.3,0.9,1.1,1.9,2.3,8.6
2,3,Bam Adebayo,C,25,MIA,72,72,35.2,8.2,15.1,...,0.803,2.4,6.9,9.3,3.3,1.2,0.8,2.6,2.8,20.7
3,4,Ochai Agbaji,SG,22,UTA,53,16,19.2,2.4,5.5,...,0.765,0.7,1.2,1.9,0.9,0.3,0.2,0.6,1.6,6.9
4,5,Santi Aldama,PF,22,MEM,73,19,22.2,3.3,6.9,...,0.745,1.1,3.7,4.8,1.2,0.6,0.6,0.8,1.9,9.2


In [4]:
print("shape", nba_22_23.shape)
for col in nba_22_23.columns:
    print(col, nba_22_23[col].dtype, end=" / ")
print()
print(nba_22_23.isna().any())

shape (661, 30)
Rk int64 / Player object / Pos object / Age int64 / Tm object / G int64 / GS int64 / MP float64 / FG float64 / FGA float64 / FG% float64 / 3P float64 / 3PA float64 / 3P% float64 / 2P float64 / 2PA float64 / 2P% float64 / eFG% float64 / FT float64 / FTA float64 / FT% float64 / ORB float64 / DRB float64 / TRB float64 / AST float64 / STL float64 / BLK float64 / TOV float64 / PF float64 / PTS float64 / 
Rk        False
Player    False
Pos       False
Age       False
Tm        False
G         False
GS        False
MP        False
FG        False
FGA       False
FG%        True
3P        False
3PA       False
3P%        True
2P        False
2PA       False
2P%        True
eFG%       True
FT        False
FTA       False
FT%        True
ORB       False
DRB       False
TRB       False
AST       False
STL       False
BLK       False
TOV       False
PF        False
PTS       False
dtype: bool


### コラム結合による新変数の作成

In [5]:
def pos_to_pos3(string):
    if string == "SG" or string == "PG":
        return "guard"
    elif string == "SF" or string == "PF":
        return "forward"
    elif string == "C":
        return "center"
    else: 
        return "other"
        
nba_22_23['Pos3'] = nba_22_23['Pos'].apply(pos_to_pos3)
nba_22_23.head()

,Rk,Player,Pos,Age,Tm,G,GS,MP,FG,FGA,...,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Pos3
0,1,Precious Achiuwa,C,23,TOR,49,11,21.1,3.4,7.2,...,1.8,4.1,5.9,1.0,0.6,0.6,1.1,1.9,8.9,center
1,2,Steven Adams,C,29,MEM,42,42,27.0,3.7,6.3,...,5.1,6.5,11.5,2.3,0.9,1.1,1.9,2.3,8.6,center
2,3,Bam Adebayo,C,25,MIA,72,72,35.2,8.2,15.1,...,2.4,6.9,9.3,3.3,1.2,0.8,2.6,2.8,20.7,center
3,4,Ochai Agbaji,SG,22,UTA,53,16,19.2,2.4,5.5,...,0.7,1.2,1.9,0.9,0.3,0.2,0.6,1.6,6.9,guard
4,5,Santi Aldama,PF,22,MEM,73,19,22.2,3.3,6.9,...,1.1,3.7,4.8,1.2,0.6,0.6,0.8,1.9,9.2,forward


### 分析を特定のポジションかつ平均得点が5点以上のプレイヤーに限定

In [6]:
nba_stat = nba_22_23[nba_22_23["Pos3"] != "other"]
nba_stat = nba_stat[nba_stat["PTS"]>=5]
nba_stat.shape

(419, 31)

### データを訓練データとテストデータに分割

In [17]:
nba_train, nba_test = train_test_split(nba_stat, test_size=0.25, random_state=2025) 

### 決定木分類の実行

In [13]:
model = DecisionTreeClassifier(max_depth=5, min_samples_split=25)

nba_train_data = pd.DataFrame()
nba_test_data = pd.DataFrame()

for col in nba_train.columns:
    if nba_train[col].dtype != "object":
        nba_train_data = pd.concat([nba_train_data, nba_train[col]], axis=1)
        nba_test_data = pd.concat([nba_test_data, nba_test[col]], axis=1)
        
nba_train_target = nba_train["Pos3"]
nba_test_target = nba_test["Pos3"]
model.fit(nba_train_data, nba_train_target)

ypred_train = model.predict(nba_train_data)
ypred_test = model.predict(nba_test_data)

### 正解率の計算

In [11]:
#正解率を計算する関数
def accuracy(acct, pred):
    correct = 0
    wrong = 0
    for i in range(len(acct)):
        if acct[i]==pred[i]:
            correct+=1
        else:
            wrong+=1
    return correct/(correct+wrong)

In [12]:
#訓練データとテストデータのの正解率の計算
print("train accuracy : ",accuracy(np.array(nba_train_target), ypred_train))
print("test accuracy : ",accuracy(np.array(nba_test_target), ypred_test))

train accuracy :  0.8439490445859873
test accuracy :  0.6666666666666666


# 2. ランダムフォレスト

### ランダムフォレスト分類の実行

In [20]:
model2 = RandomForestClassifier(max_depth=5, min_samples_split=25, n_estimators=200)
model2.fit(nba_train_data, nba_train_target)
ypred_train = model2.predict(nba_train_data)
ypred_test = model2.predict(nba_test_data)

### 正解率の計算

In [21]:
print("train accuracy : ",accuracy(np.array(nba_train_target), ypred_train))
print("test accuracy : ",accuracy(np.array(nba_test_target), ypred_test))

train accuracy :  0.8630573248407644
test accuracy :  0.780952380952381
